# SVM Optimization & Experiment Analysis

## Objective
This notebook performs hyperparameter optimization for Linear SVM and RBF SVM to improve demand-level classification performance.

The optimization process uses training data only, with time-series cross-validation to preserve chronological order and minimize data leakage.

## Scope
- Linear SVM hyperparameter optimization (`C`)
- RBF SVM hyperparameter optimization (`C`, `gamma`)
- Time-series cross-validation
- Comparison of baseline and optimized models
- Performance vs computational cost analysis
- Final model selection

## Data Split
- Training: January–April 2026
- Validation: May 2026
- Testing: June 2026

The test dataset will remain untouched until final model selection.

In [2]:
import pandas as pd
import numpy as np
import time

from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

possible_paths = [
    Path("../../datasets/interim/2026/features_dataset.parquet"),
    Path("../datasets/interim/2026/features_dataset.parquet"),
    Path("datasets/interim/2026/features_dataset.parquet"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError("features_dataset.parquet tidak ditemukan.")

df = pd.read_parquet(data_path)

print("Dataset loaded:", df.shape)
print("Dataset path:", data_path.resolve())
print("Columns:", df.columns.tolist())

Dataset loaded: (1053841, 22)
Dataset path: C:\IF SEM 5\Data Science Teori\Cooked-DS-Final-Project\datasets\interim\2026\features_dataset.parquet
Columns: ['hour', 'zone_id', 'PU_longitude', 'PU_latitude', 'PU_zone_name', 'PU_borough', 'is_airport', 'is_cbd', 'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'is_major_nyc_event', 'lag_1', 'lag_24', 'lag_168', 'roll_24', 'roll_168', 'target_pickup_count', 'demand_level', 'split']


# Linear SVM Optimization

In [3]:
FEATURES = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday",
    "is_major_nyc_event",
    "PU_longitude",
    "PU_latitude",
    "is_airport",
    "is_cbd",
    "lag_1",
    "lag_24",
    "lag_168",
    "roll_24",
    "roll_168"
]

TARGET = "demand_level"

# Training data only
train_df = df[df["split"] == "train"].copy()

# Sort chronologically
train_df = train_df.sort_values("hour").reset_index(drop=True)

# Use recent 60,000 training rows to keep optimization manageable
tuning_df = train_df.tail(60000).copy()

X_tuning = tuning_df[FEATURES]
y_tuning = tuning_df[TARGET]

# Split based on unique timestamps to prevent same-hour rows
# from being separated between training and validation folds
unique_times = np.sort(tuning_df["hour"].unique())

tscv = TimeSeriesSplit(n_splits=3)

C_values = [0.01, 0.1, 1, 10]

optimization_results = []

print("Tuning data:", tuning_df.shape)
print("Unique timestamps:", len(unique_times))
print("Testing C values:", C_values)

for C_value in C_values:

    fold_scores = []
    total_start = time.time()

    for train_time_idx, val_time_idx in tscv.split(unique_times):

        train_times = unique_times[train_time_idx]
        val_times = unique_times[val_time_idx]

        fold_train = tuning_df[
            tuning_df["hour"].isin(train_times)
        ]

        fold_val = tuning_df[
            tuning_df["hour"].isin(val_times)
        ]

        X_fold_train = fold_train[FEATURES]
        y_fold_train = fold_train[TARGET]

        X_fold_val = fold_val[FEATURES]
        y_fold_val = fold_val[TARGET]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", LinearSVC(
                C=C_value,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ))
        ])

        model.fit(X_fold_train, y_fold_train)

        y_fold_pred = model.predict(X_fold_val)

        fold_f1 = f1_score(
            y_fold_val,
            y_fold_pred,
            average="macro"
        )

        fold_scores.append(fold_f1)

    elapsed_time = time.time() - total_start

    optimization_results.append({
        "C": C_value,
        "Fold 1 F1": fold_scores[0],
        "Fold 2 F1": fold_scores[1],
        "Fold 3 F1": fold_scores[2],
        "Mean Macro F1": np.mean(fold_scores),
        "Std Macro F1": np.std(fold_scores),
        "Optimization Time (s)": elapsed_time
    })

    print(
        f"C={C_value} | "
        f"Mean Macro F1={np.mean(fold_scores):.4f} | "
        f"Time={elapsed_time:.2f}s"
    )

optimization_df = pd.DataFrame(optimization_results)

optimization_df.sort_values(
    "Mean Macro F1",
    ascending=False
).reset_index(drop=True)

Tuning data: (60000, 22)
Unique timestamps: 229
Testing C values: [0.01, 0.1, 1, 10]
C=0.01 | Mean Macro F1=0.6842 | Time=1.93s
C=0.1 | Mean Macro F1=0.7190 | Time=2.38s
C=1 | Mean Macro F1=0.7319 | Time=2.09s
C=10 | Mean Macro F1=0.7342 | Time=2.16s


,C,Fold 1 F1,Fold 2 F1,Fold 3 F1,Mean Macro F1,Std Macro F1,Optimization Time (s)
0,10.00,0.748853,0.717710,0.735984,0.734182,0.012778,2.158624
1,1.00,0.746135,0.714902,0.734739,0.731926,0.012905,2.091019
2,0.10,0.725776,0.703873,0.727255,0.718968,0.010691,2.382842
3,0.01,0.678162,0.672034,0.702262,0.684153,0.013047,1.934428


# RBF Hyperparameter Optimization

In [4]:
from sklearn.svm import SVC

# Ambil 12.000 data train terbaru secara kronologis
rbf_tuning_df = train_df.tail(12000).copy()

X_rbf_tuning = rbf_tuning_df[FEATURES]
y_rbf_tuning = rbf_tuning_df[TARGET]

# Cross validation berdasarkan urutan waktu
unique_times_rbf = np.sort(rbf_tuning_df["hour"].unique())

tscv_rbf = TimeSeriesSplit(n_splits=3)

# Kombinasi parameter yang diuji
rbf_params = [
    {"C": 1, "gamma": "scale"},
    {"C": 10, "gamma": "scale"},
    {"C": 1, "gamma": 0.01},
    {"C": 10, "gamma": 0.01}
]

rbf_optimization_results = []

print("RBF tuning data:", rbf_tuning_df.shape)
print("Parameter combinations:", len(rbf_params))

for params in rbf_params:

    fold_scores = []
    start_time = time.time()

    for train_idx, val_idx in tscv_rbf.split(unique_times_rbf):

        train_times = unique_times_rbf[train_idx]
        val_times = unique_times_rbf[val_idx]

        fold_train = rbf_tuning_df[
            rbf_tuning_df["hour"].isin(train_times)
        ]

        fold_val = rbf_tuning_df[
            rbf_tuning_df["hour"].isin(val_times)
        ]

        X_fold_train = fold_train[FEATURES]
        y_fold_train = fold_train[TARGET]

        X_fold_val = fold_val[FEATURES]
        y_fold_val = fold_val[TARGET]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                kernel="rbf",
                C=params["C"],
                gamma=params["gamma"],
                class_weight="balanced"
            ))
        ])

        model.fit(X_fold_train, y_fold_train)

        y_pred = model.predict(X_fold_val)

        fold_f1 = f1_score(
            y_fold_val,
            y_pred,
            average="macro"
        )

        fold_scores.append(fold_f1)

    elapsed = time.time() - start_time

    rbf_optimization_results.append({
        "C": params["C"],
        "gamma": params["gamma"],
        "Fold 1 F1": fold_scores[0],
        "Fold 2 F1": fold_scores[1],
        "Fold 3 F1": fold_scores[2],
        "Mean Macro F1": np.mean(fold_scores),
        "Std Macro F1": np.std(fold_scores),
        "Optimization Time (s)": elapsed
    })

    print(
        f"C={params['C']} | gamma={params['gamma']} | "
        f"Mean Macro F1={np.mean(fold_scores):.4f} | "
        f"Time={elapsed:.2f}s"
    )

rbf_optimization_df = pd.DataFrame(rbf_optimization_results)

rbf_optimization_df.sort_values(
    "Mean Macro F1",
    ascending=False
).reset_index(drop=True)

RBF tuning data: (12000, 22)
Parameter combinations: 4
C=1 | gamma=scale | Mean Macro F1=0.6388 | Time=8.51s
C=10 | gamma=scale | Mean Macro F1=0.6232 | Time=7.55s
C=1 | gamma=0.01 | Mean Macro F1=0.5720 | Time=10.36s
C=10 | gamma=0.01 | Mean Macro F1=0.7175 | Time=8.00s


,C,gamma,Fold 1 F1,Fold 2 F1,Fold 3 F1,Mean Macro F1,Std Macro F1,Optimization Time (s)
0,10,0.01,0.697100,0.721981,0.733531,0.717537,0.015201,8.001153
1,1,scale,0.702172,0.494534,0.719659,0.638789,0.102253,8.514028
2,10,scale,0.581447,0.538817,0.749213,0.623159,0.090817,7.547776
3,1,0.01,0.642995,0.456951,0.616191,0.572045,0.082117,10.363688


# Evaluasi model optimized pada Validation yang sama

In [6]:
# Recreate train and validation datasets

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()

train_df = train_df.sort_values("hour").reset_index(drop=True)
val_df = val_df.sort_values("hour").reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Target available:", TARGET in val_df.columns)

Train: (668809, 22)
Validation: (195672, 22)
Target available: True


In [7]:
# ==========================================
# FINAL VALIDATION: OPTIMIZED LINEAR vs RBF
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
import pandas as pd
import time

# ------------------------------------------
# 1. Balanced training samples
# ------------------------------------------

linear_parts = []
rbf_parts = []

for label in train_df[TARGET].unique():
    group = train_df[train_df[TARGET] == label]

    linear_parts.append(
        group.sample(n=min(50000, len(group)), random_state=42)
    )

    rbf_parts.append(
        group.sample(n=min(5000, len(group)), random_state=42)
    )

linear_train_final = pd.concat(linear_parts, ignore_index=True)
rbf_train_final = pd.concat(rbf_parts, ignore_index=True)

# ------------------------------------------
# 2. Same validation subset for both models
# ------------------------------------------

X_val_final, _, y_val_final, _ = train_test_split(
    val_df[FEATURES],
    val_df[TARGET],
    train_size=30000,
    stratify=val_df[TARGET],
    random_state=42
)

# ------------------------------------------
# 3. Optimized Linear SVM (C=10)
# ------------------------------------------

optimized_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=10,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

start = time.time()
optimized_linear.fit(
    linear_train_final[FEATURES],
    linear_train_final[TARGET]
)
linear_optimized_train_time = time.time() - start

start = time.time()
pred_linear_optimized = optimized_linear.predict(X_val_final)
linear_optimized_eval_time = time.time() - start

# ------------------------------------------
# 4. Optimized RBF SVM (C=10, gamma=0.01)
# ------------------------------------------

optimized_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma=0.01,
        class_weight="balanced"
    ))
])

start = time.time()
optimized_rbf.fit(
    rbf_train_final[FEATURES],
    rbf_train_final[TARGET]
)
rbf_optimized_train_time = time.time() - start

start = time.time()
pred_rbf_optimized = optimized_rbf.predict(X_val_final)
rbf_optimized_eval_time = time.time() - start

# ------------------------------------------
# 5. Comparison
# ------------------------------------------

optimized_comparison = pd.DataFrame([
    {
        "Model": "Optimized Linear SVM",
        "Accuracy": accuracy_score(y_val_final, pred_linear_optimized),
        "Precision Macro": precision_score(y_val_final, pred_linear_optimized, average="macro"),
        "Recall Macro": recall_score(y_val_final, pred_linear_optimized, average="macro"),
        "Macro F1": f1_score(y_val_final, pred_linear_optimized, average="macro"),
        "Training Time (s)": linear_optimized_train_time,
        "Evaluation Time (s)": linear_optimized_eval_time
    },
    {
        "Model": "Optimized RBF SVM",
        "Accuracy": accuracy_score(y_val_final, pred_rbf_optimized),
        "Precision Macro": precision_score(y_val_final, pred_rbf_optimized, average="macro"),
        "Recall Macro": recall_score(y_val_final, pred_rbf_optimized, average="macro"),
        "Macro F1": f1_score(y_val_final, pred_rbf_optimized, average="macro"),
        "Training Time (s)": rbf_optimized_train_time,
        "Evaluation Time (s)": rbf_optimized_eval_time
    }
])

optimized_comparison.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision Macro,Recall Macro,Macro F1,Training Time (s),Evaluation Time (s)
0,Optimized RBF SVM,0.755667,0.766702,0.749780,0.755814,6.660571,28.574938
1,Optimized Linear SVM,0.738867,0.732361,0.727952,0.727616,3.193551,0.007549


# Trade Off Analysis

In [8]:
# ==========================================
# BASELINE vs OPTIMIZED TRADE-OFF ANALYSIS
# ==========================================

tradeoff_df = pd.DataFrame([
    {
        "Model": "Linear SVM",
        "Version": "Baseline",
        "Accuracy": 0.739633,
        "Macro F1": 0.729920,
        "Training Time (s)": 3.13,
        "Evaluation Time (s)": 0.07
    },
    {
        "Model": "Linear SVM",
        "Version": "Optimized (C=10)",
        "Accuracy": 0.738867,
        "Macro F1": 0.727616,
        "Training Time (s)": 3.193551,
        "Evaluation Time (s)": 0.007549
    },
    {
        "Model": "RBF SVM",
        "Version": "Baseline",
        "Accuracy": 0.731233,
        "Macro F1": 0.736298,
        "Training Time (s)": 7.25,
        "Evaluation Time (s)": 28.92
    },
    {
        "Model": "RBF SVM",
        "Version": "Optimized (C=10, gamma=0.01)",
        "Accuracy": 0.755667,
        "Macro F1": 0.755814,
        "Training Time (s)": 6.660571,
        "Evaluation Time (s)": 28.574938
    }
])

tradeoff_df

,Model,Version,Accuracy,Macro F1,Training Time (s),Evaluation Time (s)
0,Linear SVM,Baseline,0.739633,0.729920,3.130000,0.070000
1,Linear SVM,Optimized (C=10),0.738867,0.727616,3.193551,0.007549
2,RBF SVM,Baseline,0.731233,0.736298,7.250000,28.920000
3,RBF SVM,"Optimized (C=10, gamma=0.01)",0.755667,0.755814,6.660571,28.574938


# Coverage@K - Zone Priority Analysis

In [9]:
import pandas as pd
import numpy as np

# Pastikan validation data tersedia
val_df = df[df["split"] == "validation"].copy()
val_df = val_df.sort_values("hour").reset_index(drop=True)

# ------------------------------------------
# 1. Predict demand level using optimized RBF
# ------------------------------------------

pred_validation_full = optimized_rbf.predict(val_df[FEATURES])

coverage_df = val_df[[
    "zone_id",
    "target_pickup_count",
    TARGET
]].copy()

coverage_df["predicted_demand_level"] = pred_validation_full

# ------------------------------------------
# 2. Convert predicted demand class to priority score
# ------------------------------------------

priority_mapping = {
    "Rendah": 1,
    "Sedang": 2,
    "Tinggi": 3
}

coverage_df["priority_score"] = (
    coverage_df["predicted_demand_level"]
    .map(priority_mapping)
)

# ------------------------------------------
# 3. Aggregate predictions and actual demand
# ------------------------------------------

zone_priority = coverage_df.groupby("zone_id").agg(
    predicted_priority=("priority_score", "sum"),
    actual_pickups=("target_pickup_count", "sum"),
    predicted_high_hours=(
        "predicted_demand_level",
        lambda x: (x == "Tinggi").sum()
    ),
    actual_high_hours=(
        TARGET,
        lambda x: (x == "Tinggi").sum()
    )
).reset_index()

# Ranking based on predicted priority
zone_priority = zone_priority.sort_values(
    "predicted_priority",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------
# 4. Calculate Coverage@K
# ------------------------------------------

total_actual_demand = zone_priority["actual_pickups"].sum()

coverage_results = []

for k in [10, 20, 50]:

    top_zones = zone_priority.head(k)

    covered_demand = top_zones["actual_pickups"].sum()

    coverage = covered_demand / total_actual_demand * 100

    coverage_results.append({
        "K": k,
        "Number of Zones": len(top_zones),
        "Covered Pickups": covered_demand,
        "Total Pickups": total_actual_demand,
        "Coverage (%)": coverage
    })

coverage_at_k_df = pd.DataFrame(coverage_results)

print("Coverage@K Results:")
display(coverage_at_k_df)

print("\nTop 20 Priority Zones:")
display(zone_priority.head(20))

Coverage@K Results:


,K,Number of Zones,Covered Pickups,Total Pickups,Coverage (%)
0,10,10,847070.0,3999128.0,21.181368
1,20,20,1650521.0,3999128.0,41.272022
2,50,50,3295448.0,3999128.0,82.404164



Top 20 Priority Zones:


,zone_id,predicted_priority,actual_pickups,predicted_high_hours,actual_high_hours
0,230,2232,109348.0,744,744
1,48,2232,87163.0,744,744
2,249,2232,88142.0,744,742
3,186,2232,110994.0,744,735
4,68,2232,97497.0,744,738
5,90,2232,59913.0,744,741
6,164,2231,75280.0,743,741
7,107,2231,73400.0,743,739
8,141,2230,89659.0,742,733
9,148,2230,55674.0,742,738


In [10]:
# DIAGNOSTIC: CHECK PREDICTION DISTRIBUTION

print("Predicted demand distribution:")
print(pd.Series(pred_validation_full).value_counts())

print("\nActual demand distribution:")
print(val_df[TARGET].value_counts())

print("\nPredicted demand percentage:")
print(
    pd.Series(pred_validation_full)
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nUnique priority scores:")
print(zone_priority["predicted_priority"].nunique())

print("\nPriority score distribution:")
print(zone_priority["predicted_priority"].value_counts().head(10))

print("\nPredicted high hours distribution:")
print(zone_priority["predicted_high_hours"].value_counts().head(10))

Predicted demand distribution:
Rendah    81106
Sedang    62309
Tinggi    52257
Name: count, dtype: int64

Actual demand distribution:
demand_level
Rendah    76650
Tinggi    61039
Sedang    57983
Name: count, dtype: int64

Predicted demand percentage:
Rendah    41.45
Sedang    31.84
Tinggi    26.71
Name: proportion, dtype: float64

Unique priority scores:
203

Priority score distribution:
predicted_priority
744     30
2232     6
745      4
1148     3
2231     2
2230     2
2229     2
2228     2
2220     2
2219     2
Name: count, dtype: int64

Predicted high hours distribution:
predicted_high_hours
0      121
1       10
744      6
6        5
2        5
10       3
14       3
743      2
742      2
741      2
Name: count, dtype: int64


# Revised Zone Ranking & Coverage@K

In [11]:
# Aggregate per zone
zone_priority_revised = coverage_df.groupby("zone_id").agg(
    predicted_high_hours=(
        "predicted_demand_level",
        lambda x: (x == "Tinggi").sum()
    ),
    predicted_medium_hours=(
        "predicted_demand_level",
        lambda x: (x == "Sedang").sum()
    ),
    predicted_priority=("priority_score", "sum"),
    actual_pickups=("target_pickup_count", "sum")
).reset_index()

# Ranking:
# 1. Predicted high-demand hours
# 2. Predicted medium-demand hours
# 3. Total predicted priority score
zone_priority_revised = zone_priority_revised.sort_values(
    by=[
        "predicted_high_hours",
        "predicted_medium_hours",
        "predicted_priority"
    ],
    ascending=False
).reset_index(drop=True)

# ------------------------------------------
# Calculate Coverage@K
# ------------------------------------------

total_actual_demand = zone_priority_revised["actual_pickups"].sum()

coverage_revised_results = []

for k in [10, 20, 50]:

    top_zones = zone_priority_revised.head(k)

    covered_demand = top_zones["actual_pickups"].sum()

    coverage = (covered_demand / total_actual_demand) * 100

    coverage_revised_results.append({
        "K": k,
        "Number of Zones": len(top_zones),
        "Covered Pickups": covered_demand,
        "Total Pickups": total_actual_demand,
        "Coverage (%)": coverage
    })

coverage_revised_df = pd.DataFrame(coverage_revised_results)

print("Revised Coverage@K:")
display(coverage_revised_df)

print("\nRevised Top 20 Priority Zones:")
display(zone_priority_revised.head(20))

Revised Coverage@K:


,K,Number of Zones,Covered Pickups,Total Pickups,Coverage (%)
0,10,10,847070.0,3999128.0,21.181368
1,20,20,1699160.0,3999128.0,42.488262
2,50,50,3481073.0,3999128.0,87.045801



Revised Top 20 Priority Zones:


,zone_id,predicted_high_hours,predicted_medium_hours,predicted_priority,actual_pickups
0,48,744,0,2232,87163.0
1,68,744,0,2232,97497.0
2,90,744,0,2232,59913.0
3,186,744,0,2232,110994.0
4,230,744,0,2232,109348.0
5,249,744,0,2232,88142.0
6,107,743,1,2231,73400.0
7,164,743,1,2231,75280.0
8,141,742,2,2230,89659.0
9,148,742,2,2230,55674.0
